# 信息论（Information Theory）

对应课程：`phases/01-math-foundations/09-information-theory`

> 信息论度量意外程度。损失函数正建立在它之上。

本 notebook 把 `information_theory.py` 里的核心函数拆开：每个函数一组中文注释，后面跟一小段可运行实验。完整打印型 demo 仍在 `information_theory.py`。

**贯穿全课的模式：** $I$ = 一次结果有多意外；$H$ = 平均意外；$H(P,Q)$ = 用 $Q$ 编码真实 $P$ 的平均代价；$D_{\mathrm{KL}}(P\|Q)$ = 多付的那部分。


## 学习目标（Learning Objectives）

- 从零算熵、交叉熵、KL，并讲清三者关系
- 推导为什么最小化交叉熵 = 最大化对数似然
- 用特征和标签的互信息给特征排序
- 把困惑度理解成语言模型面对的有效词表大小


## 0. 依赖

只用标准库，和课程允许清单一致。


In [1]:
import math


## 1. 信息量：一次事件有多意外

$$
I(p) = -\log_b p
$$

底 $b=2$ 得到比特（bit）；底 $e$ 得到奈特（nat）。$p=1$ 完全不意外，$p\to 0$ 意外趋于无穷。


In [2]:
def information_content(p, base=2):
    """I(p) = -log_b(p)。p<=0 定义为 inf；p>=1 没有意外。"""
    if p <= 0:
        return float("inf")
    if p >= 1:
        return 0.0
    return -math.log(p) / math.log(base)


print("公平硬币正面 p=0.5:", information_content(0.5), "bits")
print("掷出 6     p=1/6 :", round(information_content(1 / 6), 4), "bits")
print("千分之一事件      :", round(information_content(0.001), 4), "bits")
print("必然事件 p=1      :", information_content(1.0), "bits")
print("不可能 p=0        :", information_content(0.0))
print("同一事件用 nat    :", round(information_content(0.5, base=math.e), 4))


公平硬币正面 p=0.5: 1.0 bits
掷出 6     p=1/6 : 2.585 bits
千分之一事件      : 9.9658 bits
必然事件 p=1      : 0.0 bits
不可能 p=0        : inf
同一事件用 nat    : 0.6931


## 2. 熵：分布的平均意外

$$
H(P) = -\sum_x p(x)\log_b p(x) = \mathbb{E}_{x\sim P}[I(p(x))]
$$

均匀分布熵最大。公平硬币 $H=1$ bit；几乎确定的硬币熵接近 0。$p=0$ 的项约定为 0，不进求和。


In [3]:
def entropy(probs, base=2):
    """H(P)：对每个 p>0 累加 p * I(p)。跳过零概率，避免 0*inf。"""
    return sum(
        p * information_content(p, base)
        for p in probs if p > 0
    )


fair = [0.5, 0.5]
biased = [0.99, 0.01]
die = [1 / 6] * 6

print("公平硬币 H =", entropy(fair), "bits  (应为 1.0)")
print("偏硬币 99/1 H =", round(entropy(biased), 4), "bits")
print("公平骰子 H =", round(entropy(die), 4), "bits  (log2(6)≈2.585)")
print("公平硬币 nats =", round(entropy(fair, base=math.e), 4))
print("1 bit =", round(1 / math.log2(math.e), 4), "nats")


公平硬币 H = 1.0 bits  (应为 1.0)
偏硬币 99/1 H = 0.0808 bits
公平骰子 H = 2.585 bits  (log2(6)≈2.585)
公平硬币 nats = 0.6931
1 bit = 0.6931 nats


## 3. 交叉熵：用 $Q$ 编码真实 $P$ 的平均代价

$$
H(P,Q) = -\sum_x p(x)\log_b q(x)
$$

$P$ 是真实分布，$Q$ 是模型。$Q=P$ 时交叉熵退化为熵；任何错配都会让它变大。某个 $p_i>0$ 但 $q_i=0$ 时，编码失败，结果是 $\infty$。


In [4]:
def cross_entropy(p, q, base=2):
    """H(P,Q)。真实质量落在 q=0 上时无法编码，返回 inf。"""
    total = 0.0
    for pi, qi in zip(p, q):
        if pi > 0:
            if qi <= 0:
                return float("inf")
            total += pi * (-math.log(qi) / math.log(base))
    return total


true_dist = [0.7, 0.2, 0.1]
good_model = [0.6, 0.25, 0.15]
bad_model = [0.1, 0.1, 0.8]

print("H(true)           =", round(entropy(true_dist), 4))
print("H(true, good)     =", round(cross_entropy(true_dist, good_model), 4))
print("H(true, bad)      =", round(cross_entropy(true_dist, bad_model), 4))
print("H(true, true)     =", round(cross_entropy(true_dist, true_dist), 4), "(等于熵)")
print("Q 漏掉真实支撑     =", cross_entropy([1.0, 0.0], [0.0, 1.0]))


H(true)           = 1.1568
H(true, good)     = 1.1896
H(true, bad)      = 3.0219
H(true, true)     = 1.1568 (等于熵)
Q 漏掉真实支撑     = inf


## 4. KL 散度：多付的那部分意外

$$
D_{\mathrm{KL}}(P\|Q) = H(P,Q) - H(P) = \sum_x p(x)\log_b\frac{p(x)}{q(x)}
$$

训练时 $H(P)$ 是常数，所以最小化交叉熵等价于最小化 KL。KL **不对称**，不是距离。


In [5]:
def kl_divergence(p, q, base=2):
    """KL(P||Q) = H(P,Q) - H(P)。Q 越离谱，额外代价越大。"""
    return cross_entropy(p, q, base) - entropy(p, base)


p_uniform = [0.5, 0.5]
q_peaked = [0.9, 0.1]

kl_uq = kl_divergence(p_uniform, q_peaked)
kl_qu = kl_divergence(q_peaked, p_uniform)

print("P =", p_uniform, "  Q =", q_peaked)
print("KL(P || Q) =", round(kl_uq, 4), "bits")
print("KL(Q || P) =", round(kl_qu, 4), "bits")
print("两者不相等：KL 不是对称距离")

# 恒等式：H(P,Q) = H(P) + KL(P||Q)
true_dist = [0.7, 0.2, 0.1]
good_model = [0.6, 0.25, 0.15]
h = entropy(true_dist)
kl = kl_divergence(true_dist, good_model)
ce = cross_entropy(true_dist, good_model)
print("\nH(P)+KL =", round(h + kl, 6), "  H(P,Q) =", round(ce, 6))


P = [0.5, 0.5]   Q = [0.9, 0.1]
KL(P || Q) = 0.737 bits
KL(Q || P) = 0.531 bits
两者不相等：KL 不是对称距离

H(P)+KL = 1.189572   H(P,Q) = 1.189572


## 5. Softmax：logits 变成概率

$$
\mathrm{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}
= \frac{e^{z_i-c}}{\sum_j e^{z_j-c}},\quad c=\max z
$$

减 max 不改变概率，但挡住 `exp` 溢出。分类损失几乎总是「softmax 后再取 $-\log p_y$」。


In [6]:
def softmax(logits):
    """稳定 softmax：先减 max，再 exp 归一化。"""
    max_logit = max(logits)
    exps = [math.exp(z - max_logit) for z in logits]
    total = sum(exps)
    return [e / total for e in exps]


logits = [2.0, 1.0, 0.1]
probs = softmax(logits)
print("logits :", logits)
print("softmax:", [round(p, 4) for p in probs], "  sum =", round(sum(probs), 6))
print("大数 logits 仍正常:", [round(p, 4) for p in softmax([800.0, 801.0, 802.0])])


logits : [2.0, 1.0, 0.1]
softmax: [0.659, 0.2424, 0.0986]   sum = 1.0
大数 logits 仍正常: [0.09, 0.2447, 0.6652]


## 6. 交叉熵损失：分类里的 $-\log p_y$

one-hot 目标把 $H(P,Q)$ 收成一项：

$$
\ell = -\log q_{y} = -\log\mathrm{softmax}(z)_y
$$

模型对正确类越自信，损失越接近 0；错得很自信时 $p_y\to 0$，损失很大。单位是 nats（自然对数）。


In [7]:
def cross_entropy_loss(true_class, logits):
    """分类 CE：先 softmax，再 -log(正确类概率)。"""
    probs = softmax(logits)
    return -math.log(probs[true_class])


confident = [5.0, 0.1, 0.0]   # 正确类 logit 远大于其余
wrong = [0.0, 0.1, 5.0]       # 质量堆在错误类上
true_class = 0

print("自信且正确  logits", confident)
print("  p_true =", round(softmax(confident)[true_class], 6))
print("  CE     =", round(cross_entropy_loss(true_class, confident), 4), "nats")

print("自信但错误  logits", wrong)
print("  p_true =", round(softmax(wrong)[true_class], 6))
print("  CE     =", round(cross_entropy_loss(true_class, wrong), 4), "nats")

print("\n同一组 logits，换真实类:")
logits = [2.0, 1.0, 0.1]
probs = softmax(logits)
for c in range(3):
    print(f"  class {c}: p={probs[c]:.4f}  loss={cross_entropy_loss(c, logits):.4f}")


自信且正确  logits [5.0, 0.1, 0.0]
  p_true = 0.986014
  CE     = 0.0141 nats
自信但错误  logits [0.0, 0.1, 5.0]
  p_true = 0.006644
  CE     = 5.0141 nats

同一组 logits，换真实类:
  class 0: p=0.6590  loss=0.4170
  class 1: p=0.2424  loss=1.4170
  class 2: p=0.0986  loss=2.3170


## 7. 负对数似然：一批样本的平均 CE

$$
\mathrm{NLL} = -\frac{1}{N}\sum_{n=1}^{N}\log q_{y_n}
$$

最小化交叉熵 = 最大化训练数据在模型下的似然。`cross_entropy_loss` 对 batch 取平均就是 NLL。


In [8]:
def negative_log_likelihood(labels, all_logits):
    """batch 平均 CE。与逐样本 -log softmax(z)[y] 的均值相同。"""
    return sum(
        cross_entropy_loss(label, logits)
        for label, logits in zip(labels, all_logits)
    ) / len(labels)


labels = [0, 1, 0]
all_logits = [
    [3.0, 0.0, 0.0],   # 很对
    [0.0, 2.5, 0.1],   # 也对
    [0.0, 4.0, 0.0],   # 错得很自信
]
nll = negative_log_likelihood(labels, all_logits)
manual = -sum(
    math.log(softmax(lg)[lb]) for lb, lg in zip(labels, all_logits)
) / len(labels)
print("NLL          =", round(nll, 6))
print("手动平均 -log =", round(manual, 6))
print("差           =", abs(nll - manual))


NLL          = 1.430099
手动平均 -log = 1.430099
差           = 0.0


## 8. 困惑度：交叉熵的指数

$$
\mathrm{PPL} =
\begin{cases}
e^{\bar{\ell}} & \text{nats}\\
2^{\bar{\ell}} & \text{bits}
\end{cases}
$$

困惑度 50 表示模型平均像在 50 个等可能词里挑。均匀随机模型的困惑度等于词表大小。


In [9]:
def perplexity(avg_cross_entropy, base="e"):
    """把平均 CE 变回「有效分支数」。默认 nats -> exp。"""
    if base == "e":
        return math.exp(avg_cross_entropy)
    return 2 ** avg_cross_entropy


# 完全均匀的 4 类：平均 CE = log(4)，PPL 应为 4
uniform_logits = [0.0, 0.0, 0.0, 0.0]
ce_uniform = cross_entropy_loss(0, uniform_logits)
print("4 类均匀  CE =", round(ce_uniform, 4), "  PPL =", round(perplexity(ce_uniform), 4))

print("自信正确  PPL =", round(perplexity(cross_entropy_loss(0, [5.0, 0.0, 0.0])), 4))
print("bits 下公平硬币 H=1 -> PPL =", perplexity(1.0, base=2))


4 类均匀  CE = 1.3863   PPL = 4.0
自信正确  PPL = 1.0135
bits 下公平硬币 H=1 -> PPL = 2.0


## 9. 联合熵：两变量一起有多不确定

$$
H(X,Y) = -\sum_{x,y} p(x,y)\log_b p(x,y)
$$

独立时 $H(X,Y)=H(X)+H(Y)$；共享信息时联合熵更小，少掉的部分就是互信息。


In [10]:
def joint_entropy(joint_probs, base=2):
    """把联合分布摊平后当普通离散分布求熵。"""
    total = 0.0
    for row in joint_probs:
        for pxy in row:
            if pxy > 0:
                total -= pxy * math.log(pxy) / math.log(base)
    return total


independent = [[0.25, 0.25], [0.25, 0.25]]
dependent = [[0.45, 0.05], [0.05, 0.45]]

print("独立 2x2  H(X,Y) =", joint_entropy(independent), "bits  (应为 2)")
print("相关 2x2  H(X,Y) =", round(joint_entropy(dependent), 4), "bits  (< 2)")


独立 2x2  H(X,Y) = 2.0 bits  (应为 2)
相关 2x2  H(X,Y) = 1.469 bits  (< 2)


## 10. 条件熵：知道 $X$ 之后 $Y$ 还剩多少意外

$$
H(Y|X) = -\sum_{x,y} p(x,y)\log_b p(y|x) = H(X,Y) - H(X)
$$

$X$ 完全决定 $Y$ 时为 0；独立时 $H(Y|X)=H(Y)$。


In [11]:
def conditional_entropy(joint_probs, base=2):
    """H(Y|X)：用 p(y|x)=p(x,y)/p(x) 对联合分布求期望。"""
    rows = len(joint_probs)
    cols = len(joint_probs[0])
    margin_x = [sum(joint_probs[i][j] for j in range(cols)) for i in range(rows)]

    h_yx = 0.0
    for i in range(rows):
        for j in range(cols):
            pxy = joint_probs[i][j]
            if pxy > 0 and margin_x[i] > 0:
                p_y_given_x = pxy / margin_x[i]
                h_yx -= pxy * math.log(p_y_given_x) / math.log(base)
    return h_yx


dependent = [[0.45, 0.05], [0.05, 0.45]]
hx = entropy([sum(row) for row in dependent])
hxy = joint_entropy(dependent)
hyx = conditional_entropy(dependent)
print("相关联合:", dependent)
print("H(X)      =", round(hx, 4))
print("H(Y|X)    =", round(hyx, 4))
print("H(X)+H(Y|X) =", round(hx + hyx, 6), "  H(X,Y) =", round(hxy, 6))

independent = [[0.25, 0.25], [0.25, 0.25]]
print("\n独立时 H(Y|X) =", conditional_entropy(independent), "  H(Y) =", entropy([0.5, 0.5]))


相关联合: [[0.45, 0.05], [0.05, 0.45]]
H(X)      = 1.0
H(Y|X)    = 0.469
H(X)+H(Y|X) = 1.468996   H(X,Y) = 1.468996

独立时 H(Y|X) = 1.0   H(Y) = 1.0


## 11. 互信息：知道一个变量，另一个少了多少意外

$$
I(X;Y) = \sum_{x,y} p(x,y)\log_b\frac{p(x,y)}{p(x)p(y)}
= H(X)+H(Y)-H(X,Y)
$$

独立则 $I=0$；完全相关则 $I=H(X)=H(Y)$。它对称，和 KL 不同。


In [12]:
def mutual_information(joint_probs, base=2):
    """I(X;Y)：对每个格子累加 pxy * log(pxy / (px*py))。"""
    rows = len(joint_probs)
    cols = len(joint_probs[0])
    margin_x = [sum(joint_probs[i][j] for j in range(cols)) for i in range(rows)]
    margin_y = [sum(joint_probs[i][j] for i in range(rows)) for j in range(cols)]

    mi = 0.0
    for i in range(rows):
        for j in range(cols):
            pxy = joint_probs[i][j]
            if pxy > 0 and margin_x[i] > 0 and margin_y[j] > 0:
                mi += pxy * math.log(pxy / (margin_x[i] * margin_y[j])) / math.log(base)
    return mi


tiny = [[0.45, 0.05], [0.05, 0.45]]
print("2x2 联合:", tiny)
print("I(X;Y) 直接计算 =", round(mutual_information(tiny), 4), "bits")

hx = entropy([sum(row) for row in tiny])
hy = entropy([sum(tiny[i][j] for i in range(2)) for j in range(2)])
hxy = joint_entropy(tiny)
identity = hx + hy - hxy
print("H(X) =", round(hx, 4), "  H(Y) =", round(hy, 4), "  H(X,Y) =", round(hxy, 4))
print("H(X)+H(Y)-H(X,Y) =", round(identity, 6))
print("与 I(X;Y) 之差     =", abs(identity - mutual_information(tiny)))

print("\n独立联合 I =", mutual_information([[0.25, 0.25], [0.25, 0.25]]), "(应为 0)")
print("完全相关 [[0.5,0],[0,0.5]] I =", mutual_information([[0.5, 0.0], [0.0, 0.5]]), "(应为 1)")


2x2 联合: [[0.45, 0.05], [0.05, 0.45]]
I(X;Y) 直接计算 = 0.531 bits
H(X) = 1.0   H(Y) = 1.0   H(X,Y) = 1.469
H(X)+H(Y)-H(X,Y) = 0.531004
与 I(X;Y) 之差     = 0.0

独立联合 I = 0.0 (应为 0)
完全相关 [[0.5,0],[0,0.5]] I = 1.0 (应为 1)


## 11. 互信息选特征（学习目标）

$I(X;Y)$ 大 = 知道 X 就少了 Y 的不确定性。下面三个二元特征：强信号几乎等于标签，弱信号有噪声，纯噪声应接近 0。


In [13]:
import random

random.seed(42)
n = 200
target = [random.choice([0, 1]) for _ in range(n)]


def mi_binary(feat, y):
    joint = [[0.0, 0.0], [0.0, 0.0]]
    for a, b in zip(feat, y):
        joint[a][b] += 1.0 / n
    return mutual_information(joint)


strong = [t if random.random() > 0.1 else 1 - t for t in target]
weak = [t if random.random() > 0.35 else 1 - t for t in target]
noise = [random.choice([0, 1]) for _ in range(n)]
ranked = sorted(
    [("strong", mi_binary(strong, target)), ("weak", mi_binary(weak, target)), ("noise", mi_binary(noise, target))],
    key=lambda kv: -kv[1],
)
for name, mi in ranked:
    print(f"{name:<8} I(X;Y)={mi:.4f}")


strong   I(X;Y)=0.5860
weak     I(X;Y)=0.0693
noise    I(X;Y)=0.0110


## 对照表

| 函数 | 角色 |
|------|------|
| `information_content` | $I(p)=-\log p$，一次结果的意外 |
| `entropy` | $H(P)$，分布的平均意外 |
| `cross_entropy` | $H(P,Q)$，用 $Q$ 编码 $P$ 的平均代价 |
| `kl_divergence` | $H(P,Q)-H(P)$，多付的比特；不对称 |
| `softmax` | logits $\to$ 概率；减 max 防溢出 |
| `cross_entropy_loss` | 分类 $-\log p_y$（nats） |
| `negative_log_likelihood` | batch 平均 CE；最大似然的负号 |
| `perplexity` | $\exp(\bar{\ell})$，有效分支数 |
| `joint_entropy` | $H(X,Y)$ |
| `conditional_entropy` | $H(Y|X)=H(X,Y)-H(X)$ |
| `mutual_information` | $I(X;Y)=H(X)+H(Y)-H(X,Y)$ |

要看完整打印型 demo（含标签平滑、特征 MI 排序），运行：

```bash
python information_theory.py
```
